# E6 机械臂 VLA 云端微调与多检查点物理优选（Google Colab 最佳效果版）

同一份 E6 双视角高质量示教数据（**`320` 条** = 4 类任务各 80 条，共 **`19,200` 帧**，`eye` 全局 + 避障 `wrist` 腕部近景，`10 Hz`，含 `0.85~1.15` 灯光随机化），配合“最佳效果 + 防耗光额度”方案：

1. **路线 A：`SmolVLA`（强烈建议先跑）**：约 `4.5 亿`参数。
   - **选 `A100 (40GB)`（推荐，梯度最稳）**：`batch_size=64`、`steps=20,000`、`num_workers=4`、`use_amp=true`，约 **60～75 分钟**，消耗约 **`13～17` 计算单元（CU）**。
   - **选 `L4 (24GB)`（也完全能跑，更省 CU）**：`batch_size=32`、`steps=20,000`，约 **1.5～2 小时**，消耗约 **`8～10` CU**。
   - 训完自动评测多个检查点选出物理表现最好的模型，下载回 **Mac 16GB** 跑实时 3D 窗口闭环对比 M5！
2. **路线 B：`π0.5`（`lerobot/pi05_base` + 高秩 LoRA 进阶对比）**：约 `35 亿`参数。
   - **选 `A100 (40GB)`**：高秩 LoRA (`r=32`)、`batch_size=16`、`steps=6,000`、`bfloat16`、梯度检查点，约 **1.5～2 小时**，消耗约 **`20～26` CU**。
   - 需先在 Hugging Face 同意 `google/paligemma-3b-pt-224` 协议并准备 Read Token。
3. **两条全跑总预算**：约 **`28～43` CU**（仅占每月 `200` CU 的 `14%～22%`）。第 4 格训完自动备份到 Google Drive，第 5 格用 MuJoCo 实地考试选优，第 6 格等待 15 秒后自动 `runtime.unassign()` 释放 GPU！

> **出发前 2 步准备**：
> 1. 在 Google Drive 创建文件夹 `robot-navigation-lab`，把 Mac 上生成的 `ml/m6_vla/data/e6_colab_bundle.zip` 传进去（路径：`MyDrive/robot-navigation-lab/e6_colab_bundle.zip`）。
> 2. Colab 右上角 `更改运行时类型` 选 **`A100`**（或 L4），点菜单 **代码执行程序 -> 全部运行**。


In [ ]:
# 第 1 格：检查 GPU、挂载 Google Drive、拉取仓库并解压 e6_colab_bundle.zip
!nvidia-smi

import shutil
import subprocess
import zipfile
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

DRIVE_BASE = Path("/content/drive/MyDrive/robot-navigation-lab")
BUNDLE_ZIP = DRIVE_BASE / "e6_colab_bundle.zip"
assert BUNDLE_ZIP.is_file(), f"未找到数据包：{BUNDLE_ZIP}，请先上传到 Google Drive！"

REPO_DIR = Path("/content/robot-navigation-lab")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "https://github.com/tdgcdtd/robot-navigation-lab.git", str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)

WORK_DIR = Path("/content/work")
if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)
WORK_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(BUNDLE_ZIP, "r") as zf:
    zf.extractall(WORK_DIR)

# 将脱敏后的 e6.xml 与 7 个 STL 放到仓库指定目录，供 MuJoCo 无头仿真评测直接加载
ASSETS_DST = REPO_DIR / "ml" / "m5_robot" / "assets"
ASSETS_DST.mkdir(parents=True, exist_ok=True)
shutil.copytree(WORK_DIR / "e6_assets", ASSETS_DST, dirs_exist_ok=True)

RAW_DATA_DIR = WORK_DIR / "e6_pick"
assert (RAW_DATA_DIR / "meta.json").is_file(), "解压后缺少 e6_pick/meta.json"
print("✅ 环境与数据就绪：", RAW_DATA_DIR)


In [ ]:
# 第 2 格：选择训练路线与安装依赖
import subprocess

MODEL_CHOICE = "smolvla"  # 改这里切换路线："smolvla"（推荐先跑）或 "pi05"（进阶 A100 LoRA）
AUTO_DISCONNECT = True    # True = 跑完第 6 格后等 15 秒自动释放 GPU，避免白扣计算单元

assert MODEL_CHOICE in ("smolvla", "pi05"), f"不支持的 MODEL_CHOICE: {MODEL_CHOICE}"

extra = "smolvla" if MODEL_CHOICE == "smolvla" else "pi,peft"
subprocess.run(["pip", "install", "-q", f"lerobot[{extra}]", "mujoco", "imageio", "imageio-ffmpeg"], check=True)

if MODEL_CHOICE == "pi05":
    from huggingface_hub import login
    print("请在弹窗中输入你的 Hugging Face Read Token（需已在 HF 网站同意 google/paligemma-3b-pt-224 协议）：")
    login()

!which ffmpeg || (apt-get -qq update && apt-get -qq install -y ffmpeg)
import lerobot
print("✅ lerobot 版本:", getattr(lerobot, "__version__", "unknown"))


In [ ]:
# 第 3 格：把 320 条原始视频+轨迹转为标准 LeRobotDataset v3.0 格式（并补全分位点统计）
import json
import subprocess
from pathlib import Path

LEROBOT_ROOT = Path("/content/lerobot_e6_pick")
SCRIPT_DIR = Path("/content/robot-navigation-lab/ml/m6_vla")

subprocess.run([
    "python", str(SCRIPT_DIR / "convert_to_lerobot.py"),
    "--src", "/content/work/e6_pick",
    "--dst", str(LEROBOT_ROOT),
    "--repo-id", "local/e6_pick",
    "--overwrite",
], check=True)

info = json.loads((LEROBOT_ROOT / "meta" / "info.json").read_text())
stats = json.loads((LEROBOT_ROOT / "meta" / "stats.json").read_text())
print("✅ LeRobot info:", {k: info.get(k) for k in ("total_episodes", "total_frames", "fps", "robot_type")})
print("✅ stats keys for action:", list(stats.get("action", {}).keys()))

try:
    from lerobot.datasets.lerobot_dataset import LeRobotDataset
except ImportError:
    from lerobot.common.datasets.lerobot_dataset import LeRobotDataset
ds = LeRobotDataset("local/e6_pick", root=LEROBOT_ROOT)
print("✅ 本地 LeRobotDataset 回读验证通过:", len(ds), "帧 | keys:", list(ds[0].keys()))


In [ ]:
# 第 4 格：最佳效果微调（按显存自动配最佳 batch_size + 自动对齐相机 + finally 安全存 Drive）
import json
import shlex
import shutil
import subprocess
from pathlib import Path
import torch
from huggingface_hub import hf_hub_download

LEROBOT_ROOT = Path("/content/lerobot_e6_pick")
BASE_MODEL = "lerobot/smolvla_base" if MODEL_CHOICE == "smolvla" else "lerobot/pi05_base"
RUN_NAME = "smolvla_e6" if MODEL_CHOICE == "smolvla" else "pi05_lora_e6"
OUT_DIR = Path(f"/content/outputs/{RUN_NAME}")
DRIVE_OUT = Path("/content/drive/MyDrive/robot-navigation-lab/outputs") / RUN_NAME
DRIVE_OUT.mkdir(parents=True, exist_ok=True)

if OUT_DIR.exists():
    shutil.rmtree(OUT_DIR)

vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3) if torch.cuda.is_available() else 0.0
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
print(f"🖥️ 当前设备: {gpu_name} ({vram_gb:.1f} GB VRAM)")

# 动态读取底座模型的 config.json，避免硬编码相机键名失配
base_cfg = json.loads(Path(hf_hub_download(BASE_MODEL, "config.json")).read_text())
vis_keys = [
    k for k, v in base_cfg.get("input_features", {}).items()
    if (isinstance(v, dict) and v.get("type") == "VISUAL") or "observation.images." in k
]
assert vis_keys, f"未能从 {BASE_MODEL} 的 config.json 解析出图像输入键名！"

eye_dst = next((k for k in vis_keys if "wrist" not in k.lower()), vis_keys[0])
wrist_dst = next(
    (k for k in vis_keys if k != eye_dst and any(tag in k.lower() for tag in ("wrist", "camera2", "image2"))),
    vis_keys[1] if len(vis_keys) > 1 else None,
)
rename_map = {"observation.images.eye": eye_dst}
if wrist_dst:
    rename_map["observation.images.wrist"] = wrist_dst
empty_cameras = max(0, len(vis_keys) - len(rename_map))

print("底座期望相机:", vis_keys)
print("自动构造 rename_map:", rename_map, "| empty_cameras:", empty_cameras)

if MODEL_CHOICE == "smolvla":
    batch_size = 64 if vram_gb >= 35 else 32
    cmd = [
        "lerobot-train",
        f"--policy.path={BASE_MODEL}",
        "--dataset.repo_id=local/e6_pick",
        f"--dataset.root={LEROBOT_ROOT}",
        f"--output_dir={OUT_DIR}",
        f"--job_name={RUN_NAME}",
        "--policy.device=cuda",
        "--policy.push_to_hub=false",
        "--policy.use_amp=true",
        "--num_workers=4",
        f"--policy.empty_cameras={empty_cameras}",
        f"--rename_map={json.dumps(rename_map)}",
        f"--batch_size={batch_size}",
        "--steps=20000",
        "--save_freq=5000",
        "--log_freq=200",
        "--wandb.enable=false",
    ]
else:
    batch_size = 32 if vram_gb >= 70 else 16
    cmd = [
        "lerobot-train",
        "--policy.type=pi05",
        f"--policy.pretrained_path={BASE_MODEL}",
        "--dataset.repo_id=local/e6_pick",
        f"--dataset.root={LEROBOT_ROOT}",
        f"--output_dir={OUT_DIR}",
        f"--job_name={RUN_NAME}",
        "--policy.device=cuda",
        "--policy.push_to_hub=false",
        "--peft.method_type=LORA",
        "--peft.r=32",
        "--peft.lora_alpha=32",
        "--policy.dtype=bfloat16",
        "--policy.gradient_checkpointing=true",
        "--num_workers=4",
        f"--policy.empty_cameras={empty_cameras}",
        f"--rename_map={json.dumps(rename_map)}",
        f"--batch_size={batch_size}",
        "--steps=6000",
        "--save_freq=2000",
        "--log_freq=100",
        "--wandb.enable=false",
    ]

print("执行命令：\n" + " ".join(shlex.quote(c) for c in cmd))

def _backup_latest_checkpoint():
    ckpt_root = OUT_DIR / "checkpoints"
    if not ckpt_root.is_dir():
        print("⚠️ 尚未产生任何检查点可备份")
        return None
    target = ckpt_root / "last" / "pretrained_model"
    if not target.is_dir():
        subs = sorted(p for p in ckpt_root.iterdir() if (p / "pretrained_model").is_dir())
        if not subs:
            return None
        target = subs[-1] / "pretrained_model"
    drive_ckpt = DRIVE_OUT / "pretrained_model"
    if drive_ckpt.exists():
        shutil.rmtree(drive_ckpt)
    shutil.copytree(target, drive_ckpt)
    zip_out = shutil.make_archive(str(DRIVE_OUT / f"{RUN_NAME}_last_pretrained_model"), "zip", root_dir=drive_ckpt)
    print(f"✅ 最新检查点已备份至 Google Drive：{drive_ckpt}")
    print(f"📦 最新检查点压缩包：{zip_out}")
    return drive_ckpt

SAVED_CKPT_DIR = None
try:
    subprocess.run(cmd, check=True)
finally:
    SAVED_CKPT_DIR = _backup_latest_checkpoint()


In [ ]:
# 第 5 格：多检查点 MuJoCo 物理实测筛选（挑出成功率最高、误差最小的真冠军并打包进 Drive）
import json
import os
import shutil
import subprocess
from pathlib import Path

ckpt_root = OUT_DIR / "checkpoints"
seen_resolved = {}
if ckpt_root.is_dir():
    dirs = sorted(
        (p / "pretrained_model" for p in ckpt_root.iterdir() if (p / "pretrained_model").is_dir()),
        key=lambda p: (p.parent.name == "last", p.parent.name),
    )
    for p in dirs:
        r = p.resolve()
        if r not in seen_resolved:
            seen_resolved[r] = p

candidates = list(seen_resolved.values())[-3:]
if not candidates and SAVED_CKPT_DIR is not None and Path(SAVED_CKPT_DIR).is_dir():
    candidates = [Path(SAVED_CKPT_DIR)]

assert candidates, "未找到任何可用模型检查点，请先完成第 4 格！"
print(f"🔎 将对比评测 {len(candidates)} 个检查点：{[c.parent.name for c in candidates]}")

env_vars = os.environ.copy()
env_vars["MUJOCO_GL"] = "egl"
SCRIPT = "/content/robot-navigation-lab/ml/m6_vla/eval_vla.py"
leaderboard = []

for ckpt_dir in candidates:
    tag = ckpt_dir.parent.name
    vid_path = Path(f"/content/outputs/{RUN_NAME}_eval_{tag}.mp4")
    metrics_path = Path(f"/content/outputs/{RUN_NAME}_metrics_{tag}.json")
    print(f"\n=== 评测检查点 {tag} (20 轮未见随机场景, n_action_steps=5) ===", flush=True)
    subprocess.run([
        "python", SCRIPT,
        "--checkpoint", str(ckpt_dir),
        "--episodes", "20",
        "--seed", "1000",
        "--light-range", "0.85", "1.15",
        "--n-action-steps", "5",
        "--device", "cuda",
        "--save-video", str(vid_path),
        "--video-episodes", "4",
        "--out-json", str(metrics_path),
    ], env=env_vars, check=True)
    m = json.loads(metrics_path.read_text(encoding="utf-8"))
    leaderboard.append({
        "step": tag,
        "ckpt_dir": str(ckpt_dir),
        "video_path": str(vid_path),
        "success": m["success"],
        "episodes": m["episodes"],
        "success_rate": m["success_rate"],
        "mean_err_mm": m["mean_err_mm"],
        "mean_infer_ms": m["mean_infer_ms"],
        "per_task": m["per_task"],
    })

print("\n🏆 多检查点实测排行榜（按成功率优先、平均误差越小越好）：")
ranked = sorted(
    leaderboard,
    key=lambda x: (x["success_rate"], -(x["mean_err_mm"] if x["mean_err_mm"] is not None else 999.0)),
    reverse=True,
)
for rank, row in enumerate(ranked, 1):
    print(f"  #{rank} [{row['step']}] 成功 {row['success']}/{row['episodes']} ({row['success_rate']*100:.1f}%)"
          f" | 平均误差 {row['mean_err_mm']} mm | 推理 {row['mean_infer_ms']} ms")

best = ranked[0]
best_src = Path(best["ckpt_dir"])
best_drive_dir = DRIVE_OUT / "best_pretrained_model"
if best_drive_dir.exists():
    shutil.rmtree(best_drive_dir)
shutil.copytree(best_src, best_drive_dir)
best_zip = shutil.make_archive(str(DRIVE_OUT / f"{RUN_NAME}_best_pretrained_model"), "zip", root_dir=best_drive_dir)

best_vid_dst = DRIVE_OUT / f"{RUN_NAME}_best_eval.mp4"
if Path(best["video_path"]).is_file():
    shutil.copy2(best["video_path"], best_vid_dst)

board_path = DRIVE_OUT / f"{RUN_NAME}_all_ckpts_eval.json"
board_path.write_text(json.dumps(ranked, ensure_ascii=False, indent=2), encoding="utf-8")

print(f"\n✅ 冠军检查点 [{best['step']}] 已存入 Google Drive：{best_drive_dir}")
print(f"📦 请下载这包回 Mac 跑 3D 闭环：{best_zip}")
print(f"🎬 冠军三视角录像：{best_vid_dst}")
print(f"📊 完整榜单记录：{board_path}")


In [ ]:
# 第 6 格：安全保护——等待 15 秒后自动释放 GPU，避免白白消耗每月 200 计算单元
import time

if AUTO_DISCONNECT:
    print("⏳ 训练与优选结果已全部存入 Google Drive，15 秒后自动断开并释放 GPU 运行时...")
    time.sleep(15)
    from google.colab import runtime
    runtime.unassign()
else:
    print("ℹ️ AUTO_DISCONNECT=False，GPU 保持开机；用完请务必点击“代码执行程序 -> 断开连接并删除运行时”！")
